# NB11 — Ensemble (soft / hard / stacking)

Combines the per-example probabilities saved by NB05–09 on **Ben-Sarc binary**. Soft-voting averages
class probabilities, hard-voting takes the majority label, stacking fits a logistic meta-learner on the
**validation** probabilities and applies it to test. Runs on CPU/Mac. Auto-discovers prediction files.


In [1]:
import json, re, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
def find_repo_root():
    for c in [Path("/workspace/Sarcasm_detection"), Path.cwd()] + list(Path.cwd().resolve().parents):
        if (c/"01_data").exists(): return c.resolve()
    raise RuntimeError("repo root not found.")
ROOT=find_repo_root(); TABLES=ROOT/"04_outputs"/"tables"; PRED=ROOT/"04_outputs"/"predictions"; FIGS=ROOT/"04_outputs"/"figures"
for p in [TABLES,FIGS]: p.mkdir(parents=True,exist_ok=True)
TASK="ben_sarc_binary"
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
def collect(task, split):
    out={}
    for f in sorted(PRED.glob(f"*_{task}_{split}_predictions.csv")):
        name=f.name.replace(f"_{task}_{split}_predictions.csv","")
        try: out[name]=pd.read_csv(f).dropna(subset=["text"]).drop_duplicates("text")
        except Exception as e: print("skip",f.name,e)
    return out
def macro(df): return f1_score(df["gold_label"],df["pred_label"],average="macro",zero_division=0)
print("ROOT:",ROOT)

ROOT: /Users/sefayet/Desktop/Github/Sarcasm_detection


In [2]:
test=collect(TASK,"test"); val=collect(TASK,"val")
# Substitute weak naive FGM+AWP entries with the tuned 09b; keep pool ~24 models.
DROP_FROM_ENSEMBLE = {"07_fgm_awp", "09_FINAL_proposed"}
mods=[k for k,v in test.items() if "prob_1" in v.columns and k not in DROP_FROM_ENSEMBLE]
print("dropped from pool:", DROP_FROM_ENSEMBLE & set(test))
print("09b present in pool:", "09b_fgm_awp" in mods)
print(f"{len(mods)} models with probabilities:", mods)
# individual macro-F1
indiv={k:macro(test[k]) for k in mods}
best_single=max(indiv,key=indiv.get)
print("\nIndividual macro-F1:")
for k,v in sorted(indiv.items(),key=lambda x:-x[1]): print(f"  {v:.4f}  {k}")
print(f"\nbest single: {best_single} = {indiv[best_single]:.4f}")

dropped from pool: {'07_fgm_awp', '09_FINAL_proposed'}
09b present in pool: True
23 models with probabilities: ['04_bengali-bert', '04_mbert', '04_sagorsarker-bb', '05_baseline_banglabert', '06_banglabert', '06_mbert', '06_muril', '06_sagorsarker-bb', '06_xlm-roberta', '07_awp', '07_fgm_e05', '07_freelb_k3', '08_P0_fullft', '08_P1_fullft_fgm', '08_P2_fullft_llrd', '08_P3_fullft_fgm_llrd', '08_P4_freeze6_fgm', '08_P5_freeze6_fgm_llrd', '08_P6_maxlen192_fgm_llrd', '08_P7_fgm_llrd_ls', '08_P8_large_fgm_llrd', '08_ZIHAN_repro', '09b_fgm_awp']

Individual macro-F1:
  0.8038  09b_fgm_awp
  0.8025  05_baseline_banglabert
  0.8022  08_P6_maxlen192_fgm_llrd
  0.8003  08_P7_fgm_llrd_ls
  0.7999  08_P3_fullft_fgm_llrd
  0.7992  08_P1_fullft_fgm
  0.7991  07_fgm_e05
  0.7984  07_freelb_k3
  0.7981  06_banglabert
  0.7971  06_muril
  0.7970  08_P8_large_fgm_llrd
  0.7938  08_P4_freeze6_fgm
  0.7920  08_P2_fullft_llrd
  0.7902  07_awp
  0.7846  08_P5_freeze6_fgm_llrd
  0.7828  08_ZIHAN_repro
  0.781

In [3]:
# align on text
def aligned(coll, keys, col):
    series={k:coll[k].set_index("text")[col] for k in keys}
    idx=sorted(set.intersection(*[set(s.index) for s in series.values()]))
    return pd.DataFrame({k:series[k].loc[idx] for k in keys}), idx
Ptest,idx=aligned(test,mods,"prob_1")
gold=test[mods[0]].set_index("text")["gold_label"].loc[idx].values
preds_hard=pd.DataFrame({k:test[k].set_index("text")["pred_label"].loc[idx] for k in mods})

res=[]
# soft vote
soft=(Ptest.mean(1).values>=0.5).astype(int)
res.append(("soft_vote_all", f1_score(gold,soft,average="macro")))
# hard vote
hard=(preds_hard.mean(1).values>=0.5).astype(int)
res.append(("hard_vote_all", f1_score(gold,hard,average="macro")))
# top-K soft vote by individual VAL macro-F1 (avoids weak models dragging the ensemble down)
val_f1={}
for k in mods:
    if k in val:
        vg=val[k]; val_f1[k]=f1_score(vg["gold_label"],vg["pred_label"],average="macro",zero_division=0)
    else:
        val_f1[k]=indiv[k]
order=[k for k,_ in sorted(val_f1.items(),key=lambda x:-x[1])]
for K in [3,5]:
    topk=order[:K]
    if len(topk)<2: continue
    Pk,_=aligned(test,topk,"prob_1")
    sv=(Pk.mean(1).values>=0.5).astype(int)
    res.append((f"soft_vote_top{K}", f1_score(gold,sv,average="macro")))
# stacking (LR on val probs)
try:
    from sklearn.linear_model import LogisticRegression
    vmods=[k for k in mods if k in val and "prob_1" in val[k].columns]
    Pval,vidx=aligned(val,vmods,"prob_1")
    yval=val[vmods[0]].set_index("text")["gold_label"].loc[vidx].values
    Ptest2,tidx=aligned(test,vmods,"prob_1")
    ytest=test[vmods[0]].set_index("text")["gold_label"].loc[tidx].values
    lr=LogisticRegression(max_iter=1000).fit(Pval.values,yval)
    stk=lr.predict(Ptest2.values)
    res.append(("stacking_LR", f1_score(ytest,stk,average="macro")))
except Exception as e:
    print("stacking skipped:",e)

out=pd.DataFrame(res,columns=["ensemble","test_macro_f1"]).sort_values("test_macro_f1",ascending=False)
out["best_single"]=indiv[best_single]; out["delta_vs_best_single"]=out["test_macro_f1"]-indiv[best_single]
out.to_csv(TABLES/"11_ensemble_results.csv",index=False)
print("="*70); print("  ENSEMBLE RESULTS — Ben-Sarc binary"); print("="*70)
print(out.to_string(index=False,float_format="%.4f"))
print(f"\nNOTE: ensembles use {len(mods)} models. Gains over best single are typically small on this corpus.")

  ENSEMBLE RESULTS — Ben-Sarc binary
      ensemble  test_macro_f1  best_single  delta_vs_best_single
   stacking_LR         0.8115       0.8038                0.0077
 soft_vote_all         0.8091       0.8038                0.0053
 hard_vote_all         0.8091       0.8038                0.0053
soft_vote_top5         0.8006       0.8038               -0.0032
soft_vote_top3         0.7983       0.8038               -0.0055

NOTE: ensembles use 23 models. Gains over best single are typically small on this corpus.
